# Fine-Tune a Model for E-commerce Product Categorization

This notebook adapts the original demo to a true classification task.

## Goal
Predict a product's **top-level category** from title + brand + description.

## Base model choice
We use **`answerdotai/ModernBERT-base`** as the base model.

Why this model for this task:
- Encoder models are a strong fit for supervised classification.
- ModernBERT is built for classification/retrieval and long-context inputs.
- It is much lighter to fine-tune than decoder LLMs for category prediction.

References:
- Model card: https://huggingface.co/answerdotai/ModernBERT-base
- Paper: https://arxiv.org/abs/2412.13663
- Dataset: https://huggingface.co/datasets/Shopify/product-catalogue


## 0. Install dependencies


In [ ]:
!pip install -q -U transformers datasets accelerate evaluate scikit-learn seaborn matplotlib huggingface_hub


## 1. Imports and hardware check


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"[INFO] Torch version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device)
    total_mem = torch.cuda.get_device_properties(device).total_memory / 1e9
    print(f"[INFO] GPU: {gpu_name} ({total_mem:.1f} GB)")
else:
    print("[WARN] No CUDA GPU detected. Training will be slow on CPU.")


## 2. Config


In [ ]:
BASE_MODEL_NAME = "answerdotai/ModernBERT-base"
DATASET_NAME = "Shopify/product-catalogue"
OUTPUT_DIR = "./modernbert-ecommerce-topcat"

MAX_LENGTH = 256
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
WEIGHT_DECAY = 0.01

# For quicker Colab experiments. Set to None to use full split.
MAX_TRAIN_SAMPLES = 30000
MAX_EVAL_SAMPLES = 8000

print(f"[INFO] Base model: {BASE_MODEL_NAME}")
print(f"[INFO] Dataset: {DATASET_NAME}")


## 3. Load dataset


In [ ]:
dataset = load_dataset(DATASET_NAME)
print(dataset)
print(f"[INFO] Train rows: {len(dataset['train'])}")
print(f"[INFO] Test rows:  {len(dataset['test'])}")


## 4. Build text input and labels


In [ ]:
def top_level_category(path):
    if not path:
        return "Other"
    return path.split(">")[0].strip()


def build_record(example):
    title = (example.get("product_title") or "").strip()
    desc = (example.get("product_description") or "").strip()
    brand = (example.get("ground_truth_brand") or "").strip()

    text = f"title: {title}\nbrand: {brand}\ndescription: {desc}"
    top_cat = top_level_category(example.get("ground_truth_category", ""))

    return {
        "text": text,
        "top_category": top_cat,
    }

processed = dataset.map(build_record)

# Remove columns we do not need for text-only training.
keep_cols = {"text", "top_category"}
for split in processed.keys():
    drop_cols = [c for c in processed[split].column_names if c not in keep_cols]
    processed[split] = processed[split].remove_columns(drop_cols)

print(processed)
print("\n[INFO] Sample record:")
print(processed["train"][0])


## 5. Encode labels and optionally subsample


In [ ]:
all_labels = sorted(set(processed["train"]["top_category"]) | set(processed["test"]["top_category"]))
label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"[INFO] Number of top-level categories: {len(all_labels)}")
print(f"[INFO] Categories: {all_labels}")


def add_label_id(example):
    return {"label": label2id[example["top_category"]]}

encoded = processed.map(add_label_id)

train_ds = encoded["train"]
eval_ds = encoded["test"]

if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(train_ds):
    train_ds = train_ds.shuffle(seed=SEED).select(range(MAX_TRAIN_SAMPLES))
if MAX_EVAL_SAMPLES is not None and MAX_EVAL_SAMPLES < len(eval_ds):
    eval_ds = eval_ds.shuffle(seed=SEED).select(range(MAX_EVAL_SAMPLES))

print(f"[INFO] Train used: {len(train_ds)}")
print(f"[INFO] Eval used:  {len(eval_ds)}")


## 6. Tokenize


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_tok = train_ds.map(tokenize_batch, batched=True)
eval_tok = eval_ds.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

cols_to_keep = ["input_ids", "attention_mask", "label"]
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in cols_to_keep])
eval_tok = eval_tok.remove_columns([c for c in eval_tok.column_names if c not in cols_to_keep])

train_tok.set_format(type="torch")
eval_tok.set_format(type="torch")

print(train_tok)


## 7. Load model


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=len(all_labels),
    id2label=id2label,
    label2id=label2id,
)

print(f"[INFO] Loaded model: {BASE_MODEL_NAME}")
print(f"[INFO] Num labels: {model.config.num_labels}")


## 8. Train


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print("\n[INFO] Training complete")
print(train_result.metrics)


## 9. Evaluate


In [ ]:
metrics = trainer.evaluate()
print("[INFO] Eval metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")


## 10. Error analysis snapshot


In [ ]:
pred_output = trainer.predict(eval_tok)
preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

print(classification_report(labels, preds, target_names=[id2label[i] for i in range(len(id2label))], digits=3))

cm = confusion_matrix(labels, preds, normalize="true")
plt.figure(figsize=(11, 9))
sns.heatmap(cm, cmap="Blues", cbar=True)
plt.title("Normalized Confusion Matrix (Top-Level Categories)")
plt.xlabel("Predicted label id")
plt.ylabel("True label id")
plt.tight_layout()
plt.show()


## 11. Predict on custom products


In [ ]:
softmax = torch.nn.Softmax(dim=-1)


def predict_category(product_title, brand="", product_description="", top_k=3):
    text = f"title: {product_title}\nbrand: {brand}\ndescription: {product_description}"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits[0]
        probs = softmax(logits).cpu().numpy()

    top_ids = probs.argsort()[::-1][:top_k]
    return [(id2label[i], float(probs[i])) for i in top_ids]

examples = [
    {
        "title": "Apple iPhone 15 Pro Max 256GB Unlocked Smartphone",
        "brand": "Apple",
        "description": "A17 Pro chip, 6.7-inch display, titanium design, triple camera system.",
    },
    {
        "title": "Stainless Steel Non-Stick Frying Pan 28cm",
        "brand": "KitchenPro",
        "description": "Induction compatible, oven-safe handle, PFOA-free non-stick coating.",
    },
    {
        "title": "Organic Dry Cat Food Grain Free Salmon Recipe 5kg",
        "brand": "PetHarvest",
        "description": "High protein complete nutrition for adult cats with omega-3 and vitamins.",
    },
]

for ex in examples:
    print("=" * 80)
    print(f"TITLE: {ex['title']}")
    print(predict_category(ex["title"], ex["brand"], ex["description"], top_k=3))


## 12. Save model


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"[INFO] Saved fine-tuned model to: {OUTPUT_DIR}")


## 13. Optional: Push to Hugging Face Hub


In [ ]:
# Uncomment if you want to upload.
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.push_to_hub("modernbert-ecommerce-topcat")
# tokenizer.push_to_hub("modernbert-ecommerce-topcat")
